# Lab | Data Cleaning and Formatting

In this lab, we will be working with the customer data from an insurance company, which can be found in the CSV file located at the following link: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv


## Cargar los datos

Leo el CSV directamente desde la URL y miro las primeras filas y los nombres de columna.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

(4008, 11)


,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Vehicle Class,Total Claim Amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


# Challenge 1: Data Cleaning and Formatting

## Exercise 1: Cleaning Column Names

To ensure consistency and ease of use, standardize the column names of the dataframe. Start by taking a first look at the dataframe and identifying any column names that need to be modified. Use appropriate naming conventions and make sure that column names are descriptive and informative.

*Hint*:
- *Column names should be in lower case*
- *White spaces in column names should be replaced by `_`*
- *`st` could be replaced for `state`*

**Qué hago:**

1. `df.columns.str.lower()` pasa todos los nombres a minúsculas. El `.str` permite usar métodos de texto sobre todos los nombres a la vez, sin bucle.
2. `.str.replace(" ", "_")` cambia los espacios por guiones bajos.
3. `rename(columns={"st": "state"})` da a `st` un nombre descriptivo.

**Por qué minúsculas y guiones bajos:** con nombres como `customer_lifetime_value` no hay que acordarse de mayúsculas ni de espacios, y se evitan errores al escribirlos.

**Por qué `rename` solo para `st`:** es el único nombre que no se entiende sin contexto. Con `rename` y un diccionario cambio solo esa columna sin tocar las demás.

In [2]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
df = df.rename(columns={"st": "state"})
df.columns

Index(['customer', 'state', 'gender', 'education', 'customer_lifetime_value',
       'income', 'monthly_premium_auto', 'number_of_open_complaints',
       'policy_type', 'vehicle_class', 'total_claim_amount'],
      dtype='object')

## Exercise 2: Cleaning invalid Values

The dataset contains columns with inconsistent and incorrect values that could affect the accuracy of our analysis. Therefore, we need to clean these columns to ensure that they only contain valid data.

Note that this exercise will focus only on cleaning inconsistent values and will not involve handling null values (NaN or None).

*Hint*:
- *Gender column contains various inconsistent values such as "F", "M", "Femal", "Male", "female", which need to be standardized, for example, to "M" and "F".*
- *State abbreviations be can replaced with its full name, for example "AZ": "Arizona", "Cali": "California", "WA": "Washington"*
- *In education, "Bachelors" could be replaced by "Bachelor"*
- *In Customer Lifetime Value, delete the `%` character*
- *In vehicle class, "Sports Car", "Luxury SUV" and "Luxury Car" could be replaced by "Luxury"*

Primero miro qué valores distintos hay en cada columna de texto, para saber exactamente qué hay que corregir.

In [3]:
for column in ["gender", "state", "education", "vehicle_class"]:
    print(f"{column}: {df[column].unique()}")

gender: [nan 'F' 'M' 'Femal' 'Male' 'female']
state: ['Washington' 'Arizona' 'Nevada' 'California' 'Oregon' 'Cali' 'AZ' 'WA'
 nan]
education: ['Master' 'Bachelor' 'High School or Below' 'College' 'Bachelors' 'Doctor'
 nan]
vehicle_class: ['Four-Door Car' 'Two-Door Car' 'SUV' 'Luxury SUV' 'Sports Car'
 'Luxury Car' nan]


**Qué hago en cada columna:**

- **gender:** `replace()` con un diccionario `{valor_malo: valor_bueno}`. Solo cambia los valores que aparecen como clave; `F`, `M` y los nulos se quedan como están.
- **state:** igual, pasando las abreviaturas a nombre completo.
- **education:** `Bachelors` → `Bachelor`.
- **customer_lifetime_value:** es texto con `%` al final. `.str.replace("%", "")` lo quita (sigue siendo texto; lo convierto a número en el Ejercicio 3). Pongo `regex=False` porque busco el carácter `%` literal, no un patrón.
- **vehicle_class:** agrupo los tres tipos de lujo en `Luxury`.

**Por qué `replace` con diccionario y no `map`:** `map` pone `NaN` en todo lo que no esté en el diccionario, así que tendría que listar también los valores correctos. `replace` deja intacto lo que no menciono.

**Por qué no pasar el género a mayúsculas y quedarme con la primera letra (`str[0].str.upper()`):** funcionaría con estos datos, pero un diccionario explícito deja claro qué se ha cambiado y no convierte por accidente un valor nuevo e inesperado.

In [4]:
df["gender"] = df["gender"].replace({"Femal": "F", "female": "F", "Male": "M"})
df["state"] = df["state"].replace({"AZ": "Arizona", "Cali": "California", "WA": "Washington"})
df["education"] = df["education"].replace({"Bachelors": "Bachelor"})
df["customer_lifetime_value"] = df["customer_lifetime_value"].str.replace("%", "", regex=False)
df["vehicle_class"] = df["vehicle_class"].replace(
    {"Sports Car": "Luxury", "Luxury SUV": "Luxury", "Luxury Car": "Luxury"}
)

for column in ["gender", "state", "education", "vehicle_class"]:
    print(f"{column}: {df[column].unique()}")
df["customer_lifetime_value"].head()

gender: [nan 'F' 'M']
state: ['Washington' 'Arizona' 'Nevada' 'California' 'Oregon' nan]
education: ['Master' 'Bachelor' 'High School or Below' 'College' 'Doctor' nan]
vehicle_class: ['Four-Door Car' 'Two-Door Car' 'SUV' 'Luxury' nan]


0           NaN
1     697953.59
2    1288743.17
3     764586.18
4     536307.65
Name: customer_lifetime_value, dtype: object

## Exercise 3: Formatting data types

The data types of many columns in the dataset appear to be incorrect. This could impact the accuracy of our analysis. To ensure accurate analysis, we need to correct the data types of these columns. Please update the data types of the columns as appropriate.

It is important to note that this exercise does not involve handling null values (NaN or None).

*Hint*:
- *Customer lifetime value should be numeric*
- *Number of open complaints has an incorrect format. Look at the different values it takes with `unique()` and take the middle value. As an example, 1/5/00 should be 5. Number of open complaints is a string - remember you can use `split()` to deal with it and take the number you need. Finally, since it should be numeric, cast the column to be in its proper type.*

**customer_lifetime_value:** `pd.to_numeric()` convierte el texto (ya sin `%`) en número decimal (`float`).

**Por qué `pd.to_numeric` y no `astype(float)`:** hacen lo mismo aquí, pero `to_numeric` acepta el parámetro `errors="coerce"` para convertir en `NaN` lo que no sea un número, útil cuando aparecen valores raros. Como ya revisé los valores, no lo necesito.

**number_of_open_complaints:** viene como `1/0/00`, `1/2/00`... El número de quejas es el valor del medio.

1. `.str.split("/")` separa por `/` y da una lista: `['1', '2', '00']`.
2. `.str[1]` toma el elemento de la posición 1 (el del medio).
3. `pd.to_numeric()` lo convierte en número.

**Por qué todavía no lo paso a `int`:** hay filas vacías (nulos), y una columna de enteros normal no admite `NaN`. El enunciado deja los nulos para el Ejercicio 4, y la conversión a enteros para después de tratarlos.

In [5]:
print("Values before:", df["number_of_open_complaints"].unique())

df["customer_lifetime_value"] = pd.to_numeric(df["customer_lifetime_value"])
df["number_of_open_complaints"] = pd.to_numeric(df["number_of_open_complaints"].str.split("/").str[1])

print("Values after:", df["number_of_open_complaints"].unique())
df.dtypes

Values before: ['1/0/00' '1/2/00' '1/1/00' '1/3/00' '1/5/00' '1/4/00' nan]
Values after: [ 0.  2.  1.  3.  5.  4. nan]


customer                      object
state                         object
gender                        object
education                     object
customer_lifetime_value      float64
income                       float64
monthly_premium_auto         float64
number_of_open_complaints    float64
policy_type                   object
vehicle_class                 object
total_claim_amount           float64
dtype: object

## Exercise 4: Dealing with Null values

Identify any columns with null or missing values. Identify how many null values each column has. You can use the `isnull()` function in pandas to find columns with null values.

Decide on a strategy for handling the null values. There are several options, including:

- Drop the rows or columns with null values
- Fill the null values with a specific value (such as the column mean or median for numerical variables, and mode for categorical variables)
- Fill the null values with the previous or next value in the column
- Fill the null values based on a more complex algorithm or model (note: we haven't covered this yet)

Implement your chosen strategy to handle the null values. You can use the `fillna()` function in pandas to fill null values or `dropna()` function to drop null values.

Verify that your strategy has successfully handled the null values. You can use the `isnull()` function again to check if there are still null values in the dataset.

Remember to document your process and explain your reasoning for choosing a particular strategy for handling null values.

After formatting data types, as a last step, convert all the numeric variables to integers.

**Paso 1. Contar nulos por columna.** `isnull().sum()` suma los `True` de cada columna.

In [6]:
df.isnull().sum()

customer                     2937
state                        2937
gender                       3054
education                    2937
customer_lifetime_value      2940
income                       2937
monthly_premium_auto         2937
number_of_open_complaints    2937
policy_type                  2937
vehicle_class                2937
total_claim_amount           2937
dtype: int64

Todas las columnas tienen al menos 2937 nulos. Eso encaja con lo que vimos en el lab de pandas: hay **2937 filas completamente vacías**. Lo compruebo:

In [7]:
print("Completely empty rows:", df.isnull().all(axis=1).sum())

Completely empty rows: 2937


**Estrategia, columna a columna:**

| Caso | Nulos | Decisión | Por qué |
|---|---|---|---|
| Filas completamente vacías | 2937 | **eliminar** (`dropna(how="all")`) | no tienen ningún dato; no hay nada que rellenar |
| `gender` | 117 (≈ 11 % de los clientes) | **rellenar con `"U"`** (unknown) | rellenar con la moda (`F`) inventaría el género de 117 personas y cambiaría las proporciones. Eliminar las filas perdería un 11 % de clientes que sí tienen el resto de datos. `"U"` conserva la fila y deja claro que el dato no se conoce |
| `customer_lifetime_value` | 3 | **rellenar con la mediana** | son muy pocos. Uso la mediana y no la media porque la variable es muy asimétrica (hay valores de hasta 5,8 millones que suben la media) |

**Por qué `how="all"` y no `dropna()` a secas:** `dropna()` por defecto borra cualquier fila con **algún** nulo, y se llevaría también a los 117 clientes sin género.

In [8]:
df = df.dropna(how="all").copy()   # .copy() evita el aviso SettingWithCopyWarning al modificar columnas después
df["gender"] = df["gender"].fillna("U")
df["customer_lifetime_value"] = df["customer_lifetime_value"].fillna(df["customer_lifetime_value"].median())

df.isnull().sum()

customer                     0
state                        0
gender                       0
education                    0
customer_lifetime_value      0
income                       0
monthly_premium_auto         0
number_of_open_complaints    0
policy_type                  0
vehicle_class                0
total_claim_amount           0
dtype: int64

**Paso final: numéricas a enteros.** Ya no quedan nulos, así que puedo usar `astype(int)`.

**Por qué `round()` antes de `astype(int)`:** `astype(int)` no redondea, **corta** los decimales (2,99 pasaría a 2). Redondeando primero, 2,99 pasa a 3, que es más fiel al valor real.

**Por qué con `select_dtypes(include="number")`:** así selecciono todas las columnas numéricas a la vez sin escribir sus nombres, y no me dejo ninguna.

In [9]:
numeric_columns = df.select_dtypes(include="number").columns
df[numeric_columns] = df[numeric_columns].round().astype(int)
df.dtypes

customer                     object
state                        object
gender                       object
education                    object
customer_lifetime_value       int64
income                        int64
monthly_premium_auto          int64
number_of_open_complaints     int64
policy_type                  object
vehicle_class                object
total_claim_amount            int64
dtype: object

## Exercise 5: Dealing with duplicates

Use the `.duplicated()` method to identify any duplicate rows in the dataframe.

Decide on a strategy for handling the duplicates. Options include:
- Dropping all duplicate rows
- Keeping only the first occurrence of each duplicated row
- Keeping only the last occurrence of each duplicated row
- Dropping duplicates based on a subset of columns
- Dropping duplicates based on a specific column

Implement your chosen strategy using the `drop_duplicates()` function.

Verify that your strategy has successfully handled the duplicates by checking for duplicates again using `.duplicated()`.

Remember to document your process and explain your reasoning for choosing a particular strategy for handling duplicates.

Save the cleaned dataset to a new CSV file.

*Hint*: *after dropping duplicates, reset the index to ensure consistency*.

**Paso 1. Buscar duplicados.** `duplicated()` marca con `True` cada fila que es idéntica a otra anterior. Lo compruebo en filas completas y también solo por `customer`, que es el identificador y no debería repetirse.

In [10]:
print("Duplicated rows:", df.duplicated().sum())
print("Duplicated customer IDs:", df["customer"].duplicated().sum())

Duplicated rows: 0
Duplicated customer IDs: 0


**Resultado y estrategia:** no quedan duplicados. Los había antes: las 2937 filas vacías eran idénticas entre sí, pero ya se eliminaron en el Ejercicio 4.

Aun así aplico `drop_duplicates(keep="first")`, porque este código se puede reutilizar con otros archivos de la misma fuente. `keep="first"` conserva la primera aparición y borra las siguientes: es la opción razonable cuando las filas son idénticas, porque da igual cuál se guarde.

**Por qué `reset_index(drop=True)`:** al borrar filas, el índice queda con huecos (0, 1, 2, 5, 9...). `reset_index` lo vuelve a numerar de 0 en adelante; `drop=True` evita que el índice antiguo se guarde como una columna nueva.

**Guardar:** `to_csv(..., index=False)` guarda el resultado sin la columna del índice, que no es un dato del cliente.

In [11]:
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print("Duplicated rows after cleaning:", df.duplicated().sum())
print("Final shape:", df.shape)

df.to_csv("insurance_customers_clean.csv", index=False)
df.head()

Duplicated rows after cleaning: 0
Final shape: (1071, 11)


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,U,Master,588174,0,1000,0,Personal Auto,Four-Door Car,3
1,QZ44356,Arizona,F,Bachelor,697954,0,94,0,Personal Auto,Four-Door Car,1131
2,AI49188,Nevada,F,Bachelor,1288743,48767,108,0,Personal Auto,Two-Door Car,566
3,WW63253,California,M,Bachelor,764586,0,106,0,Corporate Auto,SUV,530
4,GA49547,Washington,M,High School or Below,536308,36357,68,0,Personal Auto,Four-Door Car,17


# Bonus: Challenge 2: creating functions on a separate `py` file

Put all the data cleaning and formatting steps into functions, and create a main function that performs all the cleaning and formatting.

Write these functions in separate .py file(s). By putting these steps into functions, we can make the code more modular and easier to maintain.

*Hint: autoreload module is a utility module in Python that allows you to automatically reload modules in the current session when changes are made to the source code. This can be useful in situations where you are actively developing code and want to see the effects of changes you make without having to constantly restart the Python interpreter or Jupyter Notebook kernel.*

**Qué he hecho:** he creado el archivo `data_cleaning.py`, en la misma carpeta que este notebook, con una función por cada ejercicio:

| Función | Ejercicio |
|---|---|
| `clean_column_names(df)` | 1 |
| `clean_invalid_values(df)` | 2 |
| `format_data_types(df)` | 3 |
| `handle_nulls(df)` | 4 |
| `handle_duplicates(df)` | 5 |
| `main(url)` | lee el CSV y llama a todas en orden |

**Por qué cada función empieza con `df = df.copy()`:** así la función no modifica el DataFrame que recibe, sino que devuelve uno nuevo. Es más seguro: puedo llamar a una función sin miedo a estropear los datos originales.

**Comprobación:** ejecuto `main()` y comparo el resultado con el `df` limpiado paso a paso en este notebook. Si `equals()` devuelve `True`, los dos caminos producen exactamente lo mismo.

In [12]:
%load_ext autoreload
%autoreload 2

from data_cleaning import main

clean_df = main()
print(clean_df.shape)
print("Same result as the notebook:", clean_df.equals(df))

(1071, 11)
Same result as the notebook: True


# Bonus: Challenge 3: Analyzing Clean and Formated Data

You have been tasked with analyzing the data to identify potential areas for improving customer retention and profitability. Your goal is to identify customers with a high policy claim amount and a low customer lifetime value.

In the Pandas Lab, we only looked at high policy claim amounts because we couldn't look into low customer lifetime values. If we had tried to work with that column, we wouldn't have been able to because customer lifetime value wasn't clean and in its proper format. So after cleaning and formatting the data, let's get some more interesting insights!

Instructions:

- Review the statistics again for total claim amount and customer lifetime value to gain an understanding of the data.
- To identify potential areas for improving customer retention and profitability, we want to focus on customers with a high policy claim amount and a low customer lifetime value. Consider customers with a high policy claim amount to be those in the top 25% of the total claim amount, and clients with a low customer lifetime value to be those in the bottom 25% of the customer lifetime value. Create a pandas DataFrame object that contains information about customers with a policy claim amount greater than the 75th percentile and a customer lifetime value in the bottom 25th percentile.
- Use DataFrame methods to calculate summary statistics about the high policy claim amount and low customer lifetime value data. To do so, select both columns of the dataframe simultaneously and pass it to the `.describe()` method. This will give you descriptive statistics, such as mean, median, standard deviation, minimum and maximum values for both columns at the same time, allowing you to compare and analyze their characteristics.

**Paso 1. Repasar las dos variables** con `describe()`.

In [13]:
df[["total_claim_amount", "customer_lifetime_value"]].describe()

,total_claim_amount,customer_lifetime_value
count,1071.000000,1.071000e+03
mean,404.980392,7.931146e+05
std,293.011429,6.426676e+05
min,0.000000,2.004350e+05
25%,202.000000,4.048535e+05
50%,355.000000,5.881740e+05
75%,533.000000,8.945210e+05
max,2893.000000,5.816655e+06


**Paso 2. Calcular los dos umbrales:** el percentil 75 de `total_claim_amount` (reclamación alta) y el percentil 25 de `customer_lifetime_value` (valor bajo).

**Paso 3. Filtrar con dos condiciones a la vez.** En pandas las condiciones se combinan con `&` (y) y cada una va entre paréntesis. **Por qué no `and`:** `and` es de Python y compara un único valor; `&` compara fila a fila dos columnas de `True/False`. Los paréntesis son necesarios porque `&` tiene más prioridad que `>` y `<`.

In [14]:
claim_75 = df["total_claim_amount"].quantile(0.75)
clv_25 = df["customer_lifetime_value"].quantile(0.25)
print(f"High claim: > {claim_75}")
print(f"Low CLV: < {clv_25}")

high_claim_low_clv = df.loc[(df["total_claim_amount"] > claim_75) & (df["customer_lifetime_value"] < clv_25)]
print(f"Customers found: {len(high_claim_low_clv)}")
high_claim_low_clv.head()

High claim: > 533.0
Low CLV: < 404853.5
Customers found: 40


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
17,OE15005,California,U,College,394524,28855,101,0,Personal Auto,SUV,647
23,TZ98966,Nevada,U,Bachelor,245019,0,73,3,Corporate Auto,Four-Door Car,554
26,US89481,California,U,Bachelor,394637,0,111,0,Personal Auto,Four-Door Car,799
38,CM61827,Oregon,U,Bachelor,309003,0,90,0,Personal Auto,Two-Door Car,648
42,IL19217,California,U,Bachelor,393901,60021,99,0,Personal Auto,Four-Door Car,883


In [15]:
high_claim_low_clv[["total_claim_amount", "customer_lifetime_value"]].describe()

,total_claim_amount,customer_lifetime_value
count,40.000000,40.000000
mean,723.725000,341577.875000
std,149.278262,56921.475939
min,538.000000,228760.000000
25%,633.750000,304220.500000
50%,685.000000,365995.500000
75%,799.000000,387365.000000
max,1186.000000,402381.000000


**Conclusiones:**

- Son 40 clientes (≈ 4 % del total) que reclaman mucho y, a la vez, aportan poco valor a lo largo de su relación con la aseguradora.
- Reclaman de media ≈ 724 (casi el doble que la media general, ≈ 405) y su valor de vida medio es ≈ 342 000, frente a ≈ 793 000 del total.
- Son el grupo menos rentable: para ellos tiene sentido revisar el precio de la póliza o las condiciones de renovación, en lugar de campañas de retención.